## Generating Time Series using a Score-based Model

In [ ]:
import functools
import pandas as pd
import numpy as np
import random

# Torch
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
# tqdm
from tqdm.notebook import tqdm, trange
# matplotlib
import matplotlib.pyplot as plt
# local
from empirical_tests import run_tests, plot_daily_returns, acf_plots_block, qq_plot
from timeseries import download_data, log_returns, normalize, denormalize, TimeSeriesDataset
from metrics import calculate_scores, real_data_loading
from util import create_file_path, plot_losses

### Download 4 Real Timeseries from Yahoo
Download daily historical data for Apple Inc. (AAPL) stock, the S&P 500 Index, the USD/JPY FX spot rate, and the Gold commodity spot price from 2001-01-01 to 2021-01-01

In [ ]:
start_date = "2001-01-01"
end_date = "2021-01-01"

 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

ts_lst = [single_stock['Close'].to_numpy(), stock_index['Close'].to_numpy(), fx_spot['Close'].to_numpy(), commodity_spot['Close'].to_numpy()]

single_stock_returns = log_returns(single_stock['Close'])
stock_index_returns = log_returns(stock_index['Close'])
fx_spot_returns = log_returns(fx_spot['Close'])
commodity_spot_returns = log_returns(commodity_spot['Close'])

lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
norm_log_ssr, mlr_ssr, slr_ssr = normalize(single_stock_returns)
norm_log_sir, mlr_sir, slr_sir = normalize(stock_index_returns)
norm_log_fxr, mlr_fxr, slr_fxr = normalize(fx_spot_returns)
norm_log_csr, mlr_csr, slr_csr = normalize(commodity_spot_returns)

In [ ]:
# Convert DataFrame to PyTorch Tensor
ssr_tensor = torch.tensor(norm_log_ssr, dtype=torch.float32)
sir_tensor = torch.tensor(norm_log_sir, dtype=torch.float32)
fxr_tensor = torch.tensor(norm_log_fxr, dtype=torch.float32)
csr_tensor = torch.tensor(norm_log_csr, dtype=torch.float32)

In [ ]:
# Create TensorDataset and DataLoader
ssr_dataset = TimeSeriesDataset(ssr_tensor, 1, 0)
sir_dataset = TimeSeriesDataset(sir_tensor, 1, 0)
fxr_dataset = TimeSeriesDataset(fxr_tensor, 1, 0)
csr_dataset = TimeSeriesDataset(csr_tensor, 1, 0)

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)
real_lst = [ssr2, sir2, fxr2, csr2]

In [ ]:
batch_size = 128

### Build the Score-based model

In [ ]:
class ScoreNet(nn.Module):
    def __init__(self, input_size, hidden_size, marginal_prob_std):
        super(ScoreNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.lrelu = nn.LeakyReLU()
        self.fc2 = nn.Linear(hidden_size, input_size)
        self.marginal_prob_std = marginal_prob_std

    def forward(self, x, t):
        x = self.fc1(x)
        x = self.lrelu(x)
        x = self.fc2(x)
        x = x / self.marginal_prob_std(t)[:, None]
        return x

In [ ]:
def marginal_prob_std(t, sigma):
    t = torch.tensor(t, device=device)
    return torch.sqrt((sigma**(2 * t) - 1.) / 2. / np.log(sigma))

In [ ]:
def diffusion_coeff(t, sigma):
    return torch.tensor(sigma**t, device=device)

In [ ]:
sigma =  25.0
marginal_prob_std_fn = functools.partial(marginal_prob_std, sigma=sigma)
diffusion_coeff_fn = functools.partial(diffusion_coeff, sigma=sigma)

In [ ]:
learning_rate = 1e-5
num_epochs = 1000
input_size = 1
hidden_size = 64
condition_size = 1
sample_len = len(single_stock_returns)

In [ ]:
def Euler_Maruyama_sampler(score_model, 
                           marginal_prob_std,
                           diffusion_coeff,
                           batch_size=64, 
                           num_steps=512,
                           eps=1e-3):
    t = torch.ones(batch_size, device=device)
    init_x = torch.randn(batch_size, 1, device=device) \
            * marginal_prob_std(t)[:, None]
    time_steps = torch.linspace(1., eps, num_steps, device=device)
    step_size = time_steps[0] - time_steps[1]
    x = init_x
    
    with torch.no_grad():
        for time_step in tqdm(time_steps):
            batch_time_step = torch.ones(batch_size, device=device) * time_step
            g = diffusion_coeff(batch_time_step)
            mean_x = x + (g**2)[:, None] \
                     * score_model(x, batch_time_step) * step_size
            x = mean_x + torch.sqrt(step_size) * g[:, None] \
                     * torch.randn_like(x)    
    return mean_x

In [ ]:
def loss_fn(model, x, marginal_prob_std, eps=1e-5):
    random_t = torch.rand(x.shape[0], device=device) * (1. - eps) + eps  
    z = torch.randn_like(x)
    std = marginal_prob_std(random_t)
    perturbed_x = x + z * std[:, None]
    score = model(perturbed_x, random_t)
    loss = torch.mean(torch.sum((score * std[:, None] + z)**2, 
                                dim=(1)))
    return loss

In [ ]:
def train_score_net(dataset, model, batch_size, num_epochs, learning_rate):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_lst = list()
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    tqdm_epoch = trange(num_epochs)
    for epoch in tqdm_epoch:
        train_loss = 0.0
        for [_, inputs]  in data_loader:
            inputs = inputs.unsqueeze(-1).to(device)
            optimizer.zero_grad()
            loss = loss_fn(model, inputs, marginal_prob_std_fn)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(data_loader)
        loss_lst.append(train_loss)
        tqdm_epoch.set_description('Train Loss: {:4f}'.format(train_loss))   

    return model, loss_lst

In [ ]:
sim_returns = list()

### Train the Single Stock Model


In [ ]:
model = ScoreNet(input_size, hidden_size, marginal_prob_std_fn).to(device)

In [ ]:
trained_model, losses = train_score_net(ssr_dataset, model, batch_size, num_epochs, learning_rate)

In [ ]:
ls_filename = 'SBssrloss.png'
plot_losses([losses], ["Training Loss"], num_epochs, ls_filename)

In [ ]:
generated_series = Euler_Maruyama_sampler(trained_model, 
                                          marginal_prob_std_fn, 
                                          diffusion_coeff_fn,
                                          sample_len)

In [ ]:
norm_sample_ssr = generated_series.squeeze(-1).cpu().detach().numpy()

In [ ]:
sample_ssr = denormalize(norm_sample_ssr, mlr_ssr, slr_ssr)

In [ ]:
sim_returns.append(sample_ssr)

### Train Stock Index Model

In [ ]:
model2 = ScoreNet(input_size, hidden_size, marginal_prob_std_fn).to(device)

In [ ]:
trained_model2, losses = train_score_net(sir_dataset, model2, batch_size, num_epochs, learning_rate)

In [ ]:
ls_filename = 'SBsirloss.png'
plot_losses([losses], ["Training Loss"], num_epochs, ls_filename)

In [ ]:
generated_series2 = Euler_Maruyama_sampler(trained_model2, 
                                          marginal_prob_std_fn, 
                                          diffusion_coeff_fn,
                                          sample_len)

In [ ]:
norm_sample_sir = generated_series2.squeeze(-1).cpu().detach().numpy()

In [ ]:
sample_sir = denormalize(norm_sample_sir, mlr_sir, slr_sir)

In [ ]:
sim_returns.append(sample_sir)

### Train FX Model

In [ ]:
model3 = ScoreNet(input_size, hidden_size, marginal_prob_std_fn).to(device)

In [ ]:
trained_model3, losses = train_score_net(fxr_dataset, model3, batch_size, num_epochs, learning_rate)

In [ ]:
ls_filename = 'SBfxrloss.png'
plot_losses([losses], ["Training Loss"], num_epochs, ls_filename)

In [ ]:
generated_series3 = Euler_Maruyama_sampler(trained_model3, 
                                          marginal_prob_std_fn, 
                                          diffusion_coeff_fn,
                                          sample_len)

In [ ]:
norm_sample_fxr = generated_series3.squeeze(-1).cpu().detach().numpy()

In [ ]:
sample_fxr = denormalize(norm_sample_fxr, mlr_fxr, slr_fxr)

In [ ]:
sim_returns.append(sample_fxr)

### Train Commodity Model

In [ ]:
model4 = ScoreNet(input_size, hidden_size, marginal_prob_std_fn).to(device)

In [ ]:
trained_model4, losses = train_score_net(csr_dataset, model4, batch_size, num_epochs, learning_rate)

In [ ]:
ls_filename = 'SBcsrloss.png'
plot_losses([losses], ["Training Loss"], num_epochs, ls_filename)

In [ ]:
generated_series4 = Euler_Maruyama_sampler(trained_model4, 
                                          marginal_prob_std_fn, 
                                          diffusion_coeff_fn,
                                          sample_len)

In [ ]:
norm_sample_csr = generated_series4.squeeze(-1).cpu().detach().numpy()

In [ ]:
sample_csr = denormalize(norm_sample_csr, mlr_csr, slr_csr)

In [ ]:
sim_returns.append(sample_csr)

#### Daily Returns

In [ ]:
sbret_filename = 'SBReturns.png'
plot_daily_returns(sim_returns, stock_index.index[:-1], sbret_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
df = run_tests(sim_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
sbacf_filename = 'SBACFs.png'
acf_plots_block(sim_returns, lst_labels, sbacf_filename)

#### QQ Plot

In [ ]:
cvae_qq_filename = 'SBEQQ.png'
qq_plot(lst_returns, sim_returns, cvae_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in sim_returns:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
sb_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
sb_df

In [ ]:
sb_df.to_latex()